# J3-03 · Parallélisme : `Send` et le map-reduce

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 3 · créneau 13h00–14h15 · ⏱️ ~1h15 (dont 40 min de pratique)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Lancer N branches en parallèle avec `Send` · 2. Agréger leurs résultats avec un **reducer** · 3. Mesurer le gain réel, et ses limites sur un **serveur LLM local unique** · 4. Borner le parallélisme (`max_concurrency`) |
| 🧱 **Fil rouge** | Pour qualifier un incident, HelpDesk-Ops interroge **trois sources à la fois** : la KB, la supervision et l'historique des tickets |
| ✅ **Prérequis** | J1-03 (reducers), J3-02 (KB indexée) |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. `Send` : une branche par élément, décidée à l'exécution

### [Théorie] Le chef de chantier qui répartit les tâches

*Analogie :* le chef de chantier ne sait qu'au matin combien de corps de métier il lui faut ; il envoie chacun sur sa tâche **en même temps**, puis fait le point à midi.

- Une fonction de routage renvoie une **liste** de `Send("noeud", charge_utile)` : LangGraph lance une exécution du nœud **par** `Send`, dans le même super-step, donc **en parallèle** ;
- chaque branche reçoit **sa** charge utile (pas l'état complet) ;
- les résultats reviennent dans une clé avec **reducer** (`operator.add`) : sans lui, `InvalidUpdateError` (vu en J1-03), puisque plusieurs branches écrivent la même clé au même moment.

| | Arête conditionnelle | `Send` |
|---|---|---|
| Nombre de branches | fixé au dessin du graphe | décidé **à l'exécution** |
| Entrée de la branche | l'état complet | une charge utile choisie |
| Usage | aiguiller | répartir (map), puis agréger (reduce) |

## 3. Trois sources, sans LLM : le gain du parallélisme

Chaque source simule **1 seconde** de latence réseau (supervision, outil de ticketing distant).

In [ ]:
import asyncio
import operator
import time
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Send

from helpdesk_ops import kb
from helpdesk_ops.reseau import RESEAU

RESEAU.configurer(latence_s=1.0)


async def source_kb(question: str) -> str:
    await asyncio.sleep(1.0)   # même latence que les autres sources, pour comparer à armes égales
    # kb.rechercher est SYNCHRONE (calcul des embeddings) : dans un nœud async, on l'envoie dans un thread,
    # sinon il gèle la boucle d'événements… et toutes les autres branches avec lui
    return await asyncio.to_thread(lambda: kb.formater(kb.rechercher(question, k=2)))


async def source_supervision(question: str) -> str:
    hote = next((m.strip(" ?.,") for m in question.lower().split() if m.startswith("srv-")), "srv-erp-02")
    return await RESEAU.ping(hote)


async def source_tickets(question: str) -> str:
    await asyncio.sleep(1.0)
    return "2 tickets ouverts sur srv-erp-02 ces 7 derniers jours (NT-0031 : lenteurs ; NT-0035 : déconnexions)"


SOURCES = {"kb": source_kb, "supervision": source_supervision, "tickets": source_tickets}

# Préchauffage : le premier appel charge les modèles d'embeddings (plusieurs secondes). Sans lui, la première
# mesure compte ce chargement : pendant la préparation, le « parallèle » sortait alors PLUS LENT que le séquentiel.
# (Autre piège rencontré : QDRANT_URL en « localhost » coûtait 2 s par requête sous Windows ; d'où 127.0.0.1.)
kb.rechercher("préchauffage")


class EtatEnquete(TypedDict):
    question: str
    constats: Annotated[list[str], operator.add]   # reducer : chaque branche AJOUTE son constat


class Mission(TypedDict):                           # charge utile d'une branche
    source: str
    question: str


def repartir(state: EtatEnquete) -> list[Send]:
    return [Send("interroger", {"source": nom, "question": state["question"]}) for nom in SOURCES]


async def interroger(mission: Mission) -> dict:
    debut = time.perf_counter()
    resultat = await SOURCES[mission["source"]](mission["question"])
    return {"constats": [f"[{mission['source']} · {time.perf_counter() - debut:.1f} s] {resultat}"]}


b = StateGraph(EtatEnquete)
b.add_node("interroger", interroger)
b.add_conditional_edges(START, repartir, ["interroger"])
b.add_edge("interroger", END)
enquete = b.compile()

from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{enquete.get_graph().draw_mermaid()}\n```"))

In [ ]:
QUESTION = "srv-erp-02 est très lent depuis ce matin pour toute la compta"

debut = time.perf_counter()
etat = await enquete.ainvoke({"question": QUESTION, "constats": []})
print(f"Parallèle  : {time.perf_counter() - debut:.1f} s")
for c in etat["constats"]:
    print("  ", c[:110])

debut = time.perf_counter()
for nom in SOURCES:
    await interroger({"source": nom, "question": QUESTION})
print(f"Séquentiel : {time.perf_counter() - debut:.1f} s")

**👀 Ce que vous devez observer :** ~1 s en parallèle contre ~3 s en séquentiel : les trois attentes réseau se **recouvrent**. C'est le cas idéal du parallélisme : des tâches qui **attendent** (réseau, disque), pas des tâches qui **calculent**.

> ⚠️ **Piège :** un appel **synchrone** dans un nœud `async` (ici `kb.rechercher`, qui calcule des embeddings) bloque la boucle d'événements et fait attendre toutes les branches. `asyncio.to_thread(...)` le déporte dans un thread.

## 4. Et avec un LLM local ? Mesurer avant de promettre

### [Théorie] Un GPU, une file d'attente

Trois branches qui appellent **le même** serveur Ollama ne s'exécutent pas forcément en parallèle : Ollama traite les requêtes selon `OLLAMA_NUM_PARALLEL` et la mémoire disponible ; au-delà, elles **attendent leur tour**. Le graphe est parallèle, le GPU ne l'est pas forcément.

> 💡 **Le « Tips » du Formateur :** pendant la préparation, des mesures lancées en même temps dans deux processus se sont **mutuellement ralenties** dans la file d'Ollama, jusqu'à faire croire à des appels de 9 minutes. Mesurez en isolant, et regardez le Gantt dans Langfuse.

In [ ]:
async def resumer(texte: str) -> str:
    reponse = await llm.ainvoke(f"Résume en une phrase, en français : {texte}")
    return reponse.content


extraits = etat["constats"]

debut = time.perf_counter()
with propagate_attributes(session_id="j3-03-parallele", tags=["jour-3", "llm-sequentiel"]):
    for e in extraits:
        await resumer(e)
sequentiel = time.perf_counter() - debut

debut = time.perf_counter()
with propagate_attributes(session_id="j3-03-parallele", tags=["jour-3", "llm-parallele"]):
    await asyncio.gather(*(resumer(e) for e in extraits))
parallele = time.perf_counter() - debut

print(f"3 résumés LLM : séquentiel {sequentiel:.0f} s · parallèle {parallele:.0f} s · gain ×{sequentiel / parallele:.2f}")

**👀 Ce que vous devez observer :** un gain **nettement plus faible** qu'avec les sources réseau (×1,5 à ×1,7 mesuré pendant la préparation sur CPU, contre ×3 pour les sources) : les requêtes se partagent le même serveur. Dans Langfuse, comparez le Gantt des deux tags : les générations « parallèles » démarrent ensemble mais **s'étirent**.

**Conséquence d'architecture :** sur un GPU local unique, on parallélise les **sources** (outils, recherches), et on garde **un** appel LLM de synthèse à la fin.

## 5. Le map-reduce complet : sources en parallèle, synthèse unique

In [ ]:
class EtatDiagnostic(EtatEnquete):
    synthese: str


async def synthetiser(state: EtatDiagnostic) -> dict:
    constats = "\n".join(state["constats"])
    reponse = await llm.ainvoke(
        "Tu es l'agent support N1 de NordTech. À partir de ces constats, et d'eux seuls, rédige un diagnostic en "
        f"3 phrases qui cite ses sources entre crochets.\n\nDemande : {state['question']}\n\nConstats :\n{constats}"
    )
    return {"synthese": reponse.content}


b = StateGraph(EtatDiagnostic)
b.add_node("interroger", interroger)
b.add_node("synthetiser", synthetiser)
b.add_conditional_edges(START, repartir, ["interroger"])
b.add_edge("interroger", "synthetiser")   # le nœud de synthèse attend TOUTES les branches
b.add_edge("synthetiser", END)
diagnostic = b.compile()

debut = time.perf_counter()
with propagate_attributes(session_id="j3-03-map-reduce", tags=["jour-3", "map-reduce"]):
    etat = await diagnostic.ainvoke({"question": QUESTION, "constats": []},
                                    config={"callbacks": [langfuse_handler], "run_name": "map-reduce"})
print(f"⏱ {time.perf_counter() - debut:.0f} s\n🤖 {etat['synthese']}")
langfuse.flush()

**👀 Ce que vous devez observer :** dans Langfuse, la trace `map-reduce` montre les trois `interroger` **côte à côte** dans le Gantt, puis **une** generation de synthèse. `synthetiser` ne démarre qu'une fois les trois branches terminées : c'est la barrière du super-step.

## 6. Borner le parallélisme : `max_concurrency`

Cent tickets à analyser, c'est cent branches. Sans limite, on sature la supervision (et on se fait bloquer) ou la file d'Ollama. `config={"max_concurrency": n}` plafonne le nombre de branches simultanées.

In [ ]:
RESEAU.configurer(latence_s=0.5)
SERVEURS = ["srv-mail-01", "srv-erp-02", "srv-vpn-01", "srv-fichiers-03"] * 2   # 8 vérifications


class EtatPing(TypedDict):
    resultats: Annotated[list[str], operator.add]


async def pinger(mission: dict) -> dict:
    return {"resultats": [await RESEAU.ping(mission["hote"])]}


b = StateGraph(EtatPing)
b.add_node("pinger", pinger)
b.add_conditional_edges(START, lambda s: [Send("pinger", {"hote": h}) for h in SERVEURS], ["pinger"])
b.add_edge("pinger", END)
balayage = b.compile()

for limite in (None, 2):
    debut = time.perf_counter()
    await balayage.ainvoke({"resultats": []}, config={"max_concurrency": limite} if limite else {})
    print(f"max_concurrency={limite!s:<5} → {time.perf_counter() - debut:.1f} s pour {len(SERVEURS)} pings de 0,5 s")
RESEAU.configurer()

**👀 Ce que vous devez observer :** ~0,5 s sans limite (tout en même temps), ~2 s avec `max_concurrency=2` (4 vagues de 2). On choisit la limite selon ce que la **cible** supporte, pas selon ce que le graphe permet.

> 🛡️ **OWASP ASI08 :** un fan-out non borné transforme une demande en tempête de requêtes. Bornez, et appliquez les protections de J2 (timeout, repli) **dans** chaque branche : une source lente ne doit pas bloquer la synthèse.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Lancer N branches | fonction de routage → `[Send("noeud", charge), …]`, `add_conditional_edges(START, f, ["noeud"])` |
| Agréger | clé `Annotated[list, operator.add]` |
| Synthétiser | une arête vers un nœud unique, qui attend toutes les branches |
| Borner | `config={"max_concurrency": n}` |
| Arbitrer | paralléliser ce qui **attend** ; sur un GPU unique, un seul LLM de synthèse |

### ✅ Quiz express

1. Pourquoi la clé `constats` a-t-elle besoin d'un reducer ?
2. Pourquoi le gain est-il fort sur les sources et faible sur les appels LLM locaux ?
3. Quand `synthetiser` démarre-t-il ?
4. À quoi sert `max_concurrency` ?

<details>
<summary>Réponses</summary>

1. Plusieurs branches écrivent la même clé dans le même super-step : sans reducer, `InvalidUpdateError`.
2. Les sources attendent (réseau) et leurs attentes se recouvrent ; les appels LLM partagent un même serveur Ollama et se mettent en file ou se partagent le calcul.
3. Quand toutes les branches `interroger` sont terminées (fin du super-step).
4. À plafonner le nombre de branches simultanées, pour ne pas saturer la cible ni la file du LLM.
</details>